In [ ]:
import numpy as np
import rasterio as rio
from tqdm import tqdm
from netCDF4 import Dataset
from affine import Affine
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FEATURE_DIR = RESULTS / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
nc_data = Dataset(DATA / 'C3S-LC-L4-LCCS-Map-300m-P1Y-2020-v2.1.1.nc', 'r')

In [11]:
print(nc_data.variables['lccs_class'].shape)

(1, 64800, 129600)


In [12]:
k = 2160 
I = int(nc_data.variables['lccs_class'].shape[1] / k) 
J = int(nc_data.variables['lccs_class'].shape[2] / k) 

index_set = []
for i in range(I):
    for j in range(J):
        index_set.append((i, j))
        
bare_count_1 = np.zeros((21600, 43200))
bare_count_2 = np.zeros((21600, 43200))
bare_count_3 = np.zeros((21600, 43200))

r = 3
h = int(k / r)
w = int(k / r)

for (a, b) in tqdm(index_set):
    # Slice the large array
    sliced_original_data = nc_data.variables['lccs_class'][0, a*k:(a+1)*k, b*k:(b+1)*k].data

    # Create masks for urban and bare land
    bare_mask_1 = (sliced_original_data == 200).astype(np.int32)
    bare_mask_2 = (sliced_original_data == 201).astype(np.int32)
    bare_mask_3 = (sliced_original_data == 202).astype(np.int32)
    
    block_sums_bare_1 = bare_mask_1.reshape(k // 3, 3, k // 3, 3).sum(axis=(1, 3))
    block_sums_bare_2 = bare_mask_2.reshape(k // 3, 3, k // 3, 3).sum(axis=(1, 3))
    block_sums_bare_3 = bare_mask_3.reshape(k // 3, 3, k // 3, 3).sum(axis=(1, 3))

    # Assign the results to the final arrays
    bare_count_1[a*h:(a+1)*h, b*w:(b+1)*w] = block_sums_bare_1
    bare_count_2[a*h:(a+1)*h, b*w:(b+1)*w] = block_sums_bare_2
    bare_count_3[a*h:(a+1)*h, b*w:(b+1)*w] = block_sums_bare_3

100%|██████████| 1800/1800 [02:13<00:00, 13.52it/s]


In [13]:
bare_count = bare_count_1 + bare_count_2 + bare_count_3

In [14]:
np.max(bare_count)

np.float64(9.0)

In [15]:
bare_2020 = np.round(100 * bare_count / 9).astype(np.int8)

In [16]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [22]:
bare_2020_1km = np.empty(country_raster_1km_.shape, dtype=np.uint8)
reproject(
    source=bare_2020,
    destination=bare_2020_1km,
    src_transform=Affine(1/120, 0.0, -180, 0.0, -1/120, 90),
    src_crs="EPSG:4326",
    dst_transform=country_raster_1km_.transform,
    dst_crs=country_raster_1km_.crs,
    resampling=Resampling.bilinear,
)

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(16922, 33844), dtype=uint8),
 Affine(1000.0, 0.0, -16922000.0,
        0.0, -1000.0, 8461000.0))

In [ ]:
with rio.open(
    FEATURE_DIR / 'bare-land-2020-1km.tif',
    "w",
    driver="GTiff",
    height=country_raster_1km_.shape[0],
    width=country_raster_1km_.shape[1],
    count=1,
    dtype=bare_2020_1km.dtype,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(bare_2020_1km, 1)